In [1]:
# Importações e setup

#from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.support.select import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.alert import Alert
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import ElementClickInterceptedException

#Bibliotecas de Sistema
from datetime import datetime
from datetime import timedelta
from datetime import date
from datetime import timezone
import time
import re
import csv
import os
import requests
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd
from contextlib import closing
from sympy import false
from pydoc import text
from sympy import true
import PyPDF2

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama

#pasta_downloads = r"C:\Users\dodonin\Downloads"
pasta_downloads = r"D:\Douglas\Downloads"

# DEBUG, sim ou não
debug = False

d:\Douglas\Reps\UNICA\.venv\Lib\site-packages\seleniumwire\thirdparty\mitmproxy\contrib\kaitaistruct\tls_client_hello.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [2]:
# Definição de perfil e tipo de processo

perfil = "SRD1CIV"
#perfil = "CAN1CIV"
#perfil = "GDO1CIV"
#perfil = "STM1CIV"

tipo_processo = "EXECUÇÃO FISCAL"

print("Perfil escolhido:", perfil)
print("Tipo de Processo:", tipo_processo)

Perfil escolhido: SRD1CIV
Tipo de Processo: EXECUÇÃO FISCAL


In [3]:
# Inicia o navegador, loga no perfil escolhido

if debug:
    print(f"[DEBUG] pasta_downloads: {pasta_downloads}")

navegador = eproc.novo_browser(pasta_downloads)

if debug:
    print("[DEBUG] Navegador iniciado.")

#configura variáveis
username = "dodonin"
password = keyring.get_password("eproc", username)
pyotop_code = "GJRGIYTCGBSGKYTEHE2TOZRUGFQTMMRQ"

#eproc.login_no_eproc_tj(navegador, username, password, pyotop_code)
eproc.login_no_eproc(navegador, username, password, pyotop_code)

if debug:
    print("[DEBUG] Login realizado.")

# Entra no perfil da Vara
eproc.entrar_no_perfil(navegador, perfil)

if debug:
    print(f"[DEBUG] Entrou no perfil: {perfil}")

Driver do Eproc importado
Perfil carregado: SRD1CIV


In [25]:
# Define funções

def pega_tabela_pagina(dados_tabela):
    tabela = navegador.find_element(By.ID, "tabelaLocalizadores")
    linhas = tabela.find_elements(By.TAG_NAME, "tr")[1:]  # Ignora o cabeçalho
    lastpage = false
    while lastpage == false:
        for linha in linhas:
            # Aguarda o carregamento do tbody da tabela antes de processar as linhas
            WebDriverWait(navegador, 10).until(
                EC.presence_of_element_located((By.XPATH, "//table[@id='tabelaLocalizadores']/tbody"))
            )
            colunas = linha.find_elements(By.TAG_NAME, "td")
            if len(colunas) >= 4:
                # Pega as três primeiras colunas e também a última
                dados_tabela.append([
                    colunas[1].text.split('\n')[0],
                    colunas[2].text.split('\n')[0],
                    colunas[3].text.split('\n')[0],
                    colunas[-1].text.split('\n')[0]
                ])
            
            lastpage = true

def pega_ultima_peticao(navegador, num_processo):
    """
    Localiza a última petição ("PET") disponível no processo informado,
    captura a requisição real feita pelo navegador (Selenium Wire) para baixar o PDF
    e salva o arquivo localmente.

    Parâmetros:
        navegador     -> instância do Selenium Wire WebDriver já autenticada no eproc.
        num_processo  -> número do processo (será usado no nome do PDF).

    Retorna:
        (evento_id, caminho_pdf) se o download foi bem-sucedido.
        (None, None) caso não haja petição ou falha no download.
    """
    import os
    import requests
    from selenium.webdriver.common.by import By
    import glob

    # === 1. Coleta todos os eventos do processo ===
    documentos_eventos = []
    eventos = navegador.find_elements(By.CLASS_NAME, "td-evento")

    for evento in eventos:
        # ID do elemento que identifica o documento
        doc_id = evento.get_dom_attribute("id")

        # Sobe até a linha da tabela e pega o número do evento
        tr_element = evento.find_element(By.XPATH, "./ancestor::tr")
        evento_id = tr_element.find_element(By.XPATH, './td[2]').text
        evento_id = ''.join(filter(str.isdigit, evento_id))  # mantém só números

        # Verifica se é documento tipo PET (Petição)
        try:
            infra_link = evento.find_element(By.XPATH, ".//*[contains(@class, 'infraLinkDoc')]")
            data_nome = infra_link.get_attribute("data-nome")
        except Exception:
            data_nome = None

        documentos_eventos.append((doc_id, data_nome, evento_id))

    # === 2. Filtra apenas documentos do tipo PET ===
    documentos_eventos_filtrados = [item for item in documentos_eventos if item[1] == "PET"]

    if not documentos_eventos_filtrados:
        print("❌ Nenhuma petição encontrada.")
        return None, None

    # === 3. Pega a petição mais recente (maior evento_id) ===
    documentos_eventos_filtrados.sort(key=lambda x: int(x[2]), reverse=True)
    documento_requerido = documentos_eventos_filtrados[0]
    evento_id = documento_requerido[2]

    # Caminho onde o PDF será salvo
    caminho_pdf = os.path.join(pasta_downloads, f"{num_processo}.pdf")

    # Localiza o elemento do documento e o link
    elemento = navegador.find_element(By.ID, documento_requerido[0])
    link = elemento.find_element(By.XPATH, ".//*[contains(@class, 'infraLinkDoc')]")

    # Limpa histórico de requests para capturar apenas a próxima
    navegador.requests.clear()

    # Clica no link para gerar a requisição ao controlador.php
    link.click()
    
    # Aguarda o carregamento do preview/divBoxPreview para garantir que o clique gerou a requisição
    WebDriverWait(navegador, 10).until(
        EC.presence_of_element_located((By.ID, "divBoxPreview"))
    )
    time.sleep(2)  # Pequeno delay extra para garantir a requisição
    # Clica no botão "open-button" para abrir o documento em nova janela/aba
    # Vai para a aba que foi aberta
    if len(navegador.window_handles) > 1:
        navegador.switch_to.window(navegador.window_handles[-1])
    try:
        timestamp_download = datetime.now().isoformat()
        navegador.find_element(By.TAG_NAME, 'body').send_keys('\t\t\n')
        time.sleep(1)
    except Exception as e:
        print("Botão 'open-button' não encontrado ou erro ao clicar:", e)
    # Fecha a nova aba/janela aberta pelo botão "open-button"
    if len(navegador.window_handles) > 1:
        # Fecha todas as abas, menos a primeira
        while len(navegador.window_handles) > 1:
            navegador.switch_to.window(navegador.window_handles[-1])
            navegador.close()
        navegador.switch_to.window(navegador.window_handles[0])
    # === 4. Procura o PDF mais recente na pasta de downloads ===

    arquivos_pdf = glob.glob(os.path.join(pasta_downloads, "*.pdf"))
    if not arquivos_pdf:
        print("❌ Nenhum PDF encontrado na pasta de downloads.")
        return evento_id, None

    # Encontra o PDF mais recente
    pdf_mais_recente = max(arquivos_pdf, key=os.path.getmtime)
    tempo_modificacao = os.path.getmtime(pdf_mais_recente)
    tempo_modificacao_dt = datetime.fromtimestamp(tempo_modificacao)
    tempo_diff = abs((tempo_modificacao_dt - datetime.fromisoformat(timestamp_download)).total_seconds())

    if tempo_diff <= 10:
        # Move o arquivo para o caminho_pdf
        os.makedirs(os.path.dirname(caminho_pdf), exist_ok=True)
        os.replace(pdf_mais_recente, caminho_pdf)
        print(f"PDF renomeado e salvo em: {caminho_pdf}")

        return evento_id, caminho_pdf
    else:
        print(f"PDF mais recente tem diferença de {tempo_diff:.2f} segundos da timestamp. Não será renomeado.")
        return evento_id, None

def extrair_texto_pdf(caminho_pdf):
    with open(caminho_pdf, 'rb') as arquivo:
        leitor = PyPDF2.PdfReader(arquivo)
        texto = ""
        for pagina in leitor.pages:
            texto += pagina.extract_text()
    return texto

def pega_texto_documento(navegador, documento):
    WebDriverWait(navegador, 20).until(
        EC.presence_of_element_located((By.ID, documento))
    )
    # 1. Localizar o elemento pelo ID
    elemento = navegador.find_element(By.ID, documento)
    # 2. Criar ActionChains para executar o mouse over
    actions = ActionChains(navegador)
    # Rolar a página para o elemento antes de mover o mouse
    navegador.execute_script("arguments[0].scrollIntoView(true); window.scrollBy(0, -150);", elemento)
    # Faz o mouseover em cima do texto link infraLinkDocumento do elemento
    link_doc = elemento.find_element(By.XPATH, ".//*[contains(@class, 'infraLinkDoc')]")
    actions.move_to_element(link_doc).perform()
    # 3. Aguardar para o hover ter efeito
    time.sleep(5)
    
    # Verifica se há uma div com a classe 'divBoxPreview' visível na página
    overlays = navegador.find_elements(By.ID, "divBoxPreview")
    visiveis = [div for div in overlays if div.is_displayed()]

    conteudo = ""
    if visiveis:
        div = overlays[0]
        # Move o foco para a div
        ActionChains(navegador).move_to_element(div).click().perform()
        # Aguarda carregar o conteúdo (ajuste o tempo se necessário)
        time.sleep(1)
        # Seleciona o texto
        conteudo = navegador.find_element(By.ID, "divBoxPreview").text
        # Clica no botão de fechar o preview, se existir
        btn_close = navegador.find_element(By.ID, "divClosePreview")
        btn_close.click()    
        time.sleep(3)

    else:
        print("Erro ao recuperar o documento")
    return conteudo

def ollama_resumo(pedido):    
    pergunta_gemma = "Considere o seguinte pedido." \
    f"{pedido}" \
    "Resuma, da maneira mais objetiva possível, o pedido. Não mencione dados pessoais, como nomes, números de documento, números de processo, valores, etc. " \
    "O resumo deve ser genérico e breve (uma frase apenas, com o mínimo de palavras possível). " \
    "Se tiver mais de um pedido, retorne uma frase para cada um." \

    resumo = ollama.chat(
        model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
        messages=[{'role': 'user', 'content': f'{pergunta_gemma}'}],    
    )

    return(resumo['message']['content'])

def verifica_tipos_de_pedidos(pedido, lista_de_pedidos):
    print("========== Verificando se é um caso de uso conhecido... ==========")
    pergunta_gemma = "Considere a seguinte lista de pedidos:" \
    f"{lista_de_pedidos}" \
    f"É possível dizer que o pedido '{pedido}' pode ser adequadamente descrito por um item dessa lista?." \
    "Se sim, retorne APENAS o texto EXATO do resumo do pedido correspondente na lista. Se não, retorne APENAS o texto 'Não'."

    resumo = ollama.chat(
        model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
        messages=[{'role': 'user', 'content': f'{pergunta_gemma}'}],    
    )

    return(resumo['message']['content'])




In [17]:
#Pega os processos novos para minutar, adiciona coluna de dias, pagina por 100

# Espera até o elemento estar presente e clicável
meus_localizadores = WebDriverWait(navegador, 20).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'i[title="Meus Localizadores"]'))
)
meus_localizadores.click()

# Aguarda algum carregamento após o clique, se necessário (exemplo: espera um painel aparecer)
# WebDriverWait(navegador, 20).until(
#     EC.visibility_of_element_located((By.ID, "id_do_painel_ou_elemento_esperado"))
# )

# Localiza o primeiro <td> que contenha "CÍVEL - MINUTAR" no texto
td_civel_minutar = WebDriverWait(navegador, 20).until(
    EC.presence_of_element_located((By.XPATH, '//td[contains(text(), "CÍVEL - MINUTAR")]'))
    #EC.presence_of_element_located((By.XPATH, '//td[contains(text(), "GABINETE - MINUTAR")]'))
    #EC.presence_of_element_located((By.XPATH, '//td[contains(text(), "GABINETE – MINUTAR")]'))    
)

# Encontra o <td> imediatamente a seguir
td_seguinte = td_civel_minutar.find_element(By.XPATH, 'following-sibling::td[1]')

# Dentro desse <td>, localiza o <a> e clica, esperando estar clicável
a_element = WebDriverWait(td_seguinte, 20).until(
    EC.element_to_be_clickable((By.TAG_NAME, 'a'))
)
a_element.click()

# Localiza e clica no label "100 processos por página"
label_100 = WebDriverWait(navegador, 20).until(
    EC.element_to_be_clickable((By.XPATH, '//label[contains(text(), "100 processos por página")]'))
)
label_100.click()

# Localiza o label com id "lbloptNdiasSituacao" e clica apenas se o checkbox estiver desmarcado
label_ndias_situacao = WebDriverWait(navegador, 20).until(
    EC.element_to_be_clickable((By.ID, "lbloptNdiasSituacao"))
)
checkbox_ndias = navegador.find_element(By.ID, "optNdiasSituacao")
if not checkbox_ndias.is_selected():
    label_ndias_situacao.click()

# Localiza o botão com id "btnConsultar" e exibe na tela
botao_consultar = navegador.find_element(By.ID, "btnConsultar")
navegador.execute_script("arguments[0].scrollIntoView();", botao_consultar)

# Tenta clicar no botão "Consultar", rolando para garantir visibilidade e tratando possíveis interceptações

try:
    botao_consultar.click()
except ElementClickInterceptedException:
    navegador.execute_script("arguments[0].scrollIntoView({block: 'center'});", botao_consultar)
    time.sleep(1)
    botao_consultar.click()

In [26]:
# Renova a tabela do perfil
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"DELETE FROM {perfil}")
    conn.commit()

# Localiza a tabela pelo id e importa as três primeiras colunas (ignorando o cabeçalho)
dados_tabela = []

pega_tabela_pagina(dados_tabela)

print(dados_tabela)

table_name = f"{perfil}"

with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()

    cursor.executemany(
        f"INSERT INTO {table_name} (num_processo, dias, tipo, dt_inclusao) VALUES (?, ?, ?, ?)",
        dados_tabela
    )
    conn.commit()

#guarda os processos no db
with sqlite3.connect("urcaciv.db") as conn:
    df_pendentes = pd.read_sql_query(f"SELECT * FROM {perfil} WHERE tipo = '{tipo_processo}'", conn)
df_pendentes = df_pendentes["num_processo"].tolist()
print("processos a minutar guardados no banco de dados.")

[['5065379-95.2024.8.21.0001', '6', 'PROCEDIMENTO COMUM CÍVEL', '20/08/2025 21:48:18'], ['5009541-70.2021.8.21.0132', '56', 'Guarda', '18/08/2025 11:10:16'], ['5004567-77.2024.8.21.0069', '4', 'EXECUÇÃO FISCAL', '26/08/2025 15:11:12'], ['5004556-48.2024.8.21.0069', '4', 'EXECUÇÃO FISCAL', '26/08/2025 15:11:12'], ['5004545-19.2024.8.21.0069', '12', 'PROCEDIMENTO COMUM CÍVEL', '18/08/2025 11:11:40'], ['5004542-98.2023.8.21.0069', '29', 'EXECUÇÃO FISCAL', '26/08/2025 14:46:23'], ['5004529-65.2024.8.21.0069', '5', 'PROCEDIMENTO COMUM CÍVEL', '25/08/2025 11:17:53'], ['5004520-40.2023.8.21.0069', '60', 'EXECUÇÃO FISCAL', '17/07/2025 18:07:11'], ['5004492-72.2023.8.21.0069', '15', 'EXECUÇÃO FISCAL', '26/08/2025 15:00:38'], ['5004482-28.2023.8.21.0069', '32', 'EXECUÇÃO FISCAL', '26/08/2025 14:45:04'], ['5004481-43.2023.8.21.0069', '29', 'EXECUÇÃO FISCAL', '18/08/2025 11:10:16'], ['5004451-08.2023.8.21.0069 ', '25', 'EXECUÇÃO FISCAL', '18/08/2025 11:08:18'], ['5004419-03.2023.8.21.0069', '43', 

In [ ]:
# Adiciona toda a lista de processos na página

# Cria a tabela, se ela não existir
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"""
        CREATE TABLE IF NOT EXISTS {perfil} (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            num_processo TEXT,
            dias INTEGER,
            tipo TEXT,
            pet TEXT,
            resumo TEXT,
            minuta TEXT,
            lote TEXT,
            dt_inclusao TEXT,
            dt_resumo TEXT
        )
    """)
    conn.commit()

# Localiza a tabela pelo id e importa as três primeiras colunas (ignorando o cabeçalho)
dados_tabela = []

pega_tabela_pagina(dados_tabela)

print(dados_tabela)

table_name = f"{perfil}"

with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    # Insere apenas se o num_processo com o mesmo dt_inclusao não existe na tabela
    for row in dados_tabela:
        num_processo = row[0]
        dt_inclusao = row[3]
        cursor.execute(
            f"SELECT 1 FROM {table_name} WHERE num_processo = ? AND dt_inclusao = ?",
            (num_processo, dt_inclusao)
        )
        if cursor.fetchone() is None:
            cursor.execute(
                f"INSERT INTO {table_name} (num_processo, dias, tipo, dt_inclusao) VALUES (?, ?, ?, ?)",
                row
            )
    conn.commit()

#guarda os processos no db
with sqlite3.connect("urcaciv.db") as conn:
    df_pendentes = pd.read_sql_query(f"SELECT * FROM {perfil} WHERE tipo = '{tipo_processo}'", conn)
df_pendentes = df_pendentes["num_processo"].tolist()
print("processos a minutar guardados no banco de dados.")

[['5000024-70.2020.8.21.0069 ', '25', 'PROCEDIMENTO COMUM CÍVEL', '26/08/2025 14:48:46'], ['5000023-76.2006.8.21.0069', '8', 'CUMPRIMENTO DE SENTENÇA', '26/08/2025 15:04:25'], ['5000021-76.2024.8.21.0069', '46', 'MANDADO DE SEGURANÇA', '18/08/2025 10:55:36'], ['5000021-57.2016.8.21.0069', '6', 'CUMPRIMENTO DE SENTENÇA', '26/08/2025 15:07:53'], ['5000021-09.2006.8.21.0069', '15', 'EXECUÇÃO DE TÍTULO EXTRAJUDICIAL', '18/08/2025 10:55:36'], ['5000014-55.2022.8.21.0069', '57', 'PROCEDIMENTO COMUM CÍVEL', '26/08/2025 16:35:32'], ['5000014-36.2014.8.21.0069', '29', 'EXECUÇÃO FISCAL', '26/08/2025 14:45:59'], ['5000013-90.2010.8.21.0069', '8', 'PROCEDIMENTO COMUM CÍVEL', '26/08/2025 15:04:25'], ['5000013-65.2025.8.21.0069', '18', 'PROCEDIMENTO COMUM CÍVEL', '18/08/2025 10:55:36'], ['5000013-03.2004.8.21.0069', '8', 'EXECUÇÃO DE TÍTULO EXTRAJUDICIAL', '26/08/2025 15:04:25'], ['5000012-71.2011.8.21.0069 ', '41', 'EXECUÇÃO FISCAL', '26/08/2025 16:26:13'], ['5000011-81.2014.8.21.0069 ', '8', 'CUMP